In [7]:
import h3
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium import GeoJson
import json
from IPython.display import display

orders = pd.read_csv('data_orders.csv')
offers = pd.read_csv('data_offers.csv')

orders['hex_id'] = orders.apply(
    lambda row: h3.latlng_to_cell(row['origin_latitude'], row['origin_longitude'], 8),
    axis=1
)

hex_counts = orders.groupby('hex_id').size().reset_index(name='count')
hex_counts = hex_counts.sort_values('count', ascending=False).reset_index(drop=True)

total_orders = hex_counts['count'].sum()
hex_counts['cumulative'] = hex_counts['count'].cumsum()
hex_counts['cumulative_pct'] = hex_counts['cumulative'] / total_orders

threshold_80 = hex_counts[hex_counts['cumulative_pct'] <= 0.80]

# Get the 23 hexagons
top_hexes = threshold_80.copy()

# Create the map centered on the average coordinates
map_center = [orders['origin_latitude'].mean(), orders['origin_longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=12)

# Color scale based on count
max_count = top_hexes['count'].max()

for _, row in top_hexes.iterrows():
    # Get hexagon boundary
    boundary = h3.cell_to_boundary(row['hex_id'])
    # Convert to [lat, lng] format for folium
    boundary = [[lat, lng] for lat, lng in boundary]
    
    # Calculate color intensity
    intensity = int(255 * (1 - row['count'] / max_count))
    color = f'#{255:02x}{intensity:02x}{intensity:02x}'
    
    folium.Polygon(
        locations=boundary,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        color='black',
        weight=1,
        tooltip=f"Fails: {row['count']}"
    ).add_to(m)

m.save('failed_orders_map.html')


display(m)

Key observations:

The data is from Reading, UK — you can see the city center clearly
The darkest red hexagon in the center (Reading town center) is the biggest hotspot with 1,497 failed orders — by far the most concentrated area
Failures radiate outward from the center, getting lighter (fewer fails) as you move away from the city core
The pattern is very compact and concentrated — 23 hexagons covering a relatively small urban area account for 80% of all failures

Why does this make sense?
City centers have the highest demand density, most complex traffic, and greatest driver/rider mismatch — exactly the conditions that lead to high ETAs and cancellations.
This map ties everything together — the geographic concentration of failures confirms that the matching system struggles most in the busiest urban areas, which also happen to be the areas with the highest ETAs we saw earlier.